# RMBG 3 — self-hosted example

This notebook runs Bria's **RMBG 3** background removal on your own GPU: it asks the Bria Engine for a short-lived package credential, installs **`rmbg-3`** from Bria's **`bria-rmbg-3`** repository, and removes the background of an image. The model weights are downloaded from Bria on the first `setup()`.

**Prerequisites**

- A Bria API key in `BRIA_API_TOKEN` — any key of an account that holds a self-hosting license for RMBG 3 (https://platform.bria.ai/outpost/licenses?model=rmbg-3.0).
- Python 3.11 or 3.12, and an NVIDIA GPU with at least 40 GB of memory (an H100 is what Bria runs it on).


## 1. Package credential

Request a short-lived credential for the **`bria-rmbg-3`** PyPI repository. The Engine issues it only for a model your account holds a license for; a `403` with `not_entitled` means the account has no license for RMBG 3.


In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN", "").strip()  # a key from a secret file ends in a newline
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

ENGINE = os.environ.get("BRIA_CLIENT_BASE_URL", "https://engine.prod.bria-api.com")
url = f"{ENGINE}/v2/outpost/access/code_artifact?repository=bria-rmbg-3"
request = Request(url, headers={"api_token": BRIA_API_TOKEN})

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    body = exc.read().decode("utf-8", errors="replace")
    try:
        code = json.loads(body).get("error", {}).get("message")
    except json.JSONDecodeError:
        code = None
    if code in ("not_entitled", "entitlement_revoked"):
        raise RuntimeError(
            "This account has no active license for RMBG 3. Get one at https://platform.bria.ai/outpost/licenses?model=rmbg-3.0"
        ) from exc
    raise RuntimeError(f"Package credential request failed with HTTP {exc.code}: {body or exc.reason}") from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"The response did not include an authorization token: {payload}")
print("Package credential acquired. Expires:", result.get("expiration"))

## 2. Install dependencies

Install **`rmbg-3`** from the **`bria-rmbg-3`** repository, with the fibo-2 packages it runs on. Everything else comes from PyPI.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# pip calls os.getcwd(): if Jupyter's working folder was deleted, move to the home folder first.
try:
    Path.cwd()
except FileNotFoundError:
    os.chdir(Path.home())

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
rmbg_3_index = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-rmbg-3/simple/"
)


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(install_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [sys.executable, "-m", "pip", "install", "--upgrade", "rmbg-3>=0.1.0", "--extra-index-url", rmbg_3_index]
)

## 3. Check the license and load the model

`setup()` first checks your license with the Bria Engine, then downloads the RMBG 3 weights from Bria into `~/.bria/models/rmbg-3.0/<version>/` (set `BRIA_HOME` to move them) and verifies them; later runs reuse them. If the license is missing or has ended, `setup()` raises `LicenseError` with a link to your licenses page and loads nothing. Then it loads the model onto the GPU and runs one small warm-up.

If the install cell changed core runtime packages, restart the kernel, run the cell of section 1 again (it sets `BRIA_API_TOKEN`), then run this cell.


In [ ]:
import logging
from pathlib import Path

from common.outpost import LicenseError

from rmbg_3 import RMBG3, RMBG3Config, RMBG3Input

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

pipeline = RMBG3(config=RMBG3Config(api_token=BRIA_API_TOKEN))
try:
    pipeline.setup()
except LicenseError as error:
    raise RuntimeError(f"{error} ({error.code})") from None
print("RMBG 3 pipeline setup complete.")


## 4. Remove a background

`execute()` takes a list of images (URLs, base64 strings, numpy arrays or PIL images) and returns each one as an RGBA image: the original pixels, with the background transparent. `return_mask=True` also returns the grayscale mask (white = foreground). The same image always gets the same cutout.


In [ ]:
from IPython.display import display

IMAGE = "https://bria-test-images.s3.us-east-1.amazonaws.com/strawberry.jpg"

(result,) = pipeline.execute([RMBG3Input(image=IMAGE, return_mask=True)])
cutout_path = outputs_dir / "cutout.png"
result.image.save(cutout_path)
result.mask.save(outputs_dir / "mask.png")
print(f"Cutout {result.image.size}, saved to {cutout_path}")
display(result.image)


## 5. Cleanup

Release the GPU memory that the pipeline holds when you are done.


In [ ]:
pipeline.cleanup()
print("Cleanup done.")
